In [1]:
import sys
sys.path.append(f"./../")

import matplotlib.pyplot as plt
import numpy as np
import os
import networkx as nx
import gc
import psutil
from datetime import datetime
from contextlib import contextmanager
import itertools

from src.graphs import StaticGraph, IntersectingEdgesGraph, MultiEdgeGraph
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator, SparsePauliOp, Pauli
from scipy.linalg import expm

In [2]:
def get_memory_usage():
    """Get current memory usage in MB."""
    process = psutil.Process()
    return process.memory_info().rss / 1024 / 1024

def log_progress(message, include_memory=True):
    """Log progress with timestamp and optional memory usage."""
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    if include_memory:
        mem_usage = get_memory_usage()
        print(f"[{timestamp}] {message} (Memory: {mem_usage:.2f} MB)")
    else:
        print(f"[{timestamp}] {message}")

@contextmanager
def memory_cleanup():
    """Context manager for memory cleanup."""
    try:
        yield
    finally:
        gc.collect()

In [3]:
def graph_to_bitstring_edges(graph):
    """Convert graph to bitstring edges."""
    num_nodes = len(graph.nodes)
    num_bits = len(bin(num_nodes - 1)) - 2
    
    node_to_bitstring = {
        node: format(node, f'0{num_bits}b') 
        for node in graph.nodes
    }
    
    edges_bitstring = {
        (node_to_bitstring[u], node_to_bitstring[v]) 
        for u, v in graph.edges
    }
    
    del node_to_bitstring
    return edges_bitstring

def unitary_to_pauli(U):
    """Convert unitary matrix to Pauli operators."""
    log_progress("Starting Pauli decomposition")
    n = int(np.log2(U.shape[0]))
    dim = 2**n
    pauli_strings = []
    coeffs = []
    
    with memory_cleanup():
        for pauli_string in [''.join(p) for p in itertools.product('IXYZ', repeat=n)]:
            P = Pauli(pauli_string)
            P_op = Operator(P).data
            coeff = np.trace(P_op.conj().T @ U) / dim
            if not np.isclose(coeff, 0, atol=1e-10):
                pauli_strings.append(pauli_string)
                coeffs.append(coeff)
            del P_op
    
    return SparsePauliOp(pauli_strings, coeffs)

def construct_time_evolution_matrix(pauli_terms, delta_t):
    """Construct time evolution matrix."""
    log_progress("Constructing evolution matrix")
    matrix = Operator(pauli_terms).data
    exponent = -1j * delta_t * matrix
    return Operator(expm(exponent))

def count_gates_dynamic_walk(edges, T, delta_t):
    """Count gates for dynamic walk with cleanup."""
    log_progress("Starting dynamic walk calculation")
    
    with memory_cleanup():
        try:
            static_G = StaticGraph(edges)
            intersecting_G = IntersectingEdgesGraph(edges)
            
            big_qc = QuantumCircuit(static_G.n_qubits)
            for subgraph in intersecting_G.subgraphs:
                with memory_cleanup():
                    G = MultiEdgeGraph(subgraph.edges)
                    sub_qc = G.get_qc(simplified=True)
                    big_qc = big_qc.compose(sub_qc)
                    del G, sub_qc
            
            transpiled_qc = transpile(big_qc, basis_gates=['cx', 'u3'], 
                                    optimization_level=3, seed_transpiler=42)
            gate_counts = transpiled_qc.count_ops()
            
            log_progress(f"Dynamic circuit compiled - CX: {gate_counts.get('cx', 0)}, U3: {gate_counts.get('u3', 0)}")
            return gate_counts.get('cx', 0), gate_counts.get('u3', 0), transpiled_qc.depth()
            
        finally:
            for var in ['static_G', 'intersecting_G', 'big_qc', 'transpiled_qc']:
                if var in locals():
                    del locals()[var]

def count_gates_pauli_decomp(edges, delta_t):
    """Count gates for Pauli decomposition with cleanup."""
    log_progress("Starting Pauli decomposition calculation")
    
    with memory_cleanup():
        try:
            static_G = StaticGraph(edges)
            pauli_terms = unitary_to_pauli(static_G.get_adj_mat())
            evo_matrix = construct_time_evolution_matrix(pauli_terms, delta_t)
            
            qc = QuantumCircuit(static_G.n_qubits)
            qc.unitary(evo_matrix, range(static_G.n_qubits), label='Evo')
            
            transpiled_qc = transpile(qc, basis_gates=['cx', 'u3'],
                                    optimization_level=3, seed_transpiler=42)
            
            gate_counts = transpiled_qc.count_ops()
            log_progress(f"Pauli circuit compiled - CX: {gate_counts.get('cx', 0)}, U3: {gate_counts.get('u3', 0)}")
            return gate_counts.get('cx', 0), gate_counts.get('u3', 0), transpiled_qc.depth()
            
        finally:
            for var in ['static_G', 'pauli_terms', 'evo_matrix', 'qc', 'transpiled_qc']:
                if var in locals():
                    del locals()[var]

def process_single_graph(edges, results, error_graphs, index):
    """Process a single graph with proper cleanup and detailed logging."""
    try:
        with memory_cleanup():
            log_progress(f"\nProcessing graph {index}")
            
            # Get dynamic counts
            dynamic_counts = count_gates_dynamic_walk(edges, T, delta_t)
            dynamic_cx, dynamic_u3, dynamic_depth = dynamic_counts
            dynamic_total = dynamic_cx + dynamic_u3
            log_progress(f"Dynamic method - CX: {dynamic_cx}, U3: {dynamic_u3}, Total: {dynamic_total}, Depth: {dynamic_depth}")
            
            # Get Pauli counts
            pauli_counts = count_gates_pauli_decomp(edges, delta_t)
            pauli_cx, pauli_u3, pauli_depth = pauli_counts
            pauli_total = pauli_cx + pauli_u3
            log_progress(f"Pauli method  - CX: {pauli_cx}, U3: {pauli_u3}, Total: {pauli_total}, Depth: {pauli_depth}")
            
            if dynamic_counts and pauli_counts:
                difference = dynamic_total - pauli_total
                depth_diff = dynamic_depth - pauli_depth
                
                # Determine the winner
                result = "Win" if difference < 0 else "Lose" if difference > 0 else "Draw"
                
                log_progress(f"Result: {result}")
                log_progress(f"Gate difference (Dynamic - Pauli): {difference}")
                log_progress(f"Depth difference (Dynamic - Pauli): {depth_diff}")
                
                results.append({
                    'Graph': index,
                    'Dynamic': dynamic_total,
                    'Dynamic_CX': dynamic_cx,
                    'Dynamic_U3': dynamic_u3,
                    'Dynamic_Depth': dynamic_depth,
                    'Pauli': pauli_total,
                    'Pauli_CX': pauli_cx,
                    'Pauli_U3': pauli_u3,
                    'Pauli_Depth': pauli_depth,
                    'Difference': difference,
                    'Depth_Difference': depth_diff,
                    'Edges': edges
                })
                
    except Exception as e:
        error_graphs.append((index, edges))
        log_progress(f"Error processing graph {index}: {str(e)}")

def process_graphs_in_batches(g6_file, n_vertex, batch_size=10):
    """Process graphs in batches with memory management."""
    results = []
    error_graphs = []
    total_graphs = sum(1 for line in open(g6_file))
    
    log_progress(f"Processing {total_graphs} graphs in batches of {batch_size}")
    
    with open(g6_file, 'r') as file:
        for i, lines in enumerate(itertools.zip_longest(*[file] * batch_size)):
            batch = [line.strip() for line in lines if line and line.strip()]
            
            log_progress(f"\nProcessing batch {i+1}")
            with memory_cleanup():
                for j, g6_string in enumerate(batch):
                    try:
                        graph = nx.from_graph6_bytes(g6_string.encode('utf-8'))
                        edges = graph_to_bitstring_edges(graph)
                        del graph
                        
                        process_single_graph(edges, results, error_graphs, i * batch_size + j)
                        del edges
                        
                    except Exception as e:
                        log_progress(f"Error in batch {i}, graph {j}: {str(e)}")
                        continue
            
            log_progress(f"Batch {i+1} complete")
    
    return results, error_graphs


In [4]:
def save_graphs(graphs, filename):
    """Save graphs in G6 format with memory cleanup."""
    log_progress(f"Saving graphs to {filename}")
    with memory_cleanup():
        with open(filename, 'w') as f:
            for graph in graphs:
                G = nx.Graph(graph['Edges'])
                g6_string = nx.to_graph6_bytes(G, header=False).decode().strip()
                f.write(f"{g6_string}\n")
                del G

def save_error_graphs(error_graphs, filename):
    """Save error graphs in G6 format with memory cleanup."""
    log_progress(f"Saving error graphs to {filename}")
    with memory_cleanup():
        with open(filename, 'w') as f:
            for _, edges in error_graphs:
                G = nx.Graph(edges)
                g6_string = nx.to_graph6_bytes(G, header=False).decode().strip()
                f.write(f"{g6_string}\n")
                del G

def save_results(results, error_graphs, n_vertex):
    """Save results in simple G6 format with memory management and summary logging."""
    log_progress("\nStarting to save results")
    
    with memory_cleanup():
        # Classify graphs
        positive_graphs = [r for r in results if r['Difference'] > 0]
        negative_graphs = [r for r in results if r['Difference'] < 0]
        zero_graphs = [r for r in results if r['Difference'] == 0]
        
        # Log summary statistics
        total_graphs = len(results) + len(error_graphs)
        log_progress("\nFinal Results Summary:")
        log_progress(f"Total graphs processed: {total_graphs}")
        log_progress(f"Successful: {len(results)} ({len(results)/total_graphs*100:.1f}%)")
        log_progress(f"Errors: {len(error_graphs)} ({len(error_graphs)/total_graphs*100:.1f}%)")
        log_progress(f"Wins (Dynamic < Pauli): {len(negative_graphs)} ({len(negative_graphs)/total_graphs*100:.1f}%)")
        log_progress(f"Losses (Dynamic > Pauli): {len(positive_graphs)} ({len(positive_graphs)/total_graphs*100:.1f}%)")
        log_progress(f"Draws (Dynamic = Pauli): {len(zero_graphs)} ({len(zero_graphs)/total_graphs*100:.1f}%)")
        
        # Calculate and log averages
        if results:
            avg_dynamic = sum(r['Dynamic'] for r in results) / len(results)
            avg_pauli = sum(r['Pauli'] for r in results) / len(results)
            avg_diff = sum(r['Difference'] for r in results) / len(results)
            avg_dynamic_depth = sum(r['Dynamic_Depth'] for r in results) / len(results)
            avg_pauli_depth = sum(r['Pauli_Depth'] for r in results) / len(results)
            
            log_progress("\nAverages:")
            log_progress(f"Average Dynamic gates: {avg_dynamic:.1f}")
            log_progress(f"Average Pauli gates: {avg_pauli:.1f}")
            log_progress(f"Average difference: {avg_diff:.1f}")
            log_progress(f"Average Dynamic depth: {avg_dynamic_depth:.1f}")
            log_progress(f"Average Pauli depth: {avg_pauli_depth:.1f}")
        
        # Save each category
        save_graphs(positive_graphs, f'../data/graphs/positive_{n_vertex}cgraphs.g6')
        save_graphs(negative_graphs, f'../data/graphs/negative_{n_vertex}cgraphs.g6')
        save_graphs(zero_graphs, f'../data/graphs/zero_{n_vertex}cgraphs.g6')
        save_error_graphs(error_graphs, f'../data/graphs/error_{n_vertex}cgraphs.g6')
        
        del positive_graphs, negative_graphs, zero_graphs

def plot_results(results, error_graphs, n_vertex):
    """Plot analysis results with memory efficiency."""
    log_progress("\nGenerating plots")
    
    os.makedirs('../data/plots', exist_ok=True)
    
    with memory_cleanup():
        total_graphs = len(results) + len(error_graphs)
        
        # Calculate counts
        positive_graphs = [r for r in results if r['Difference'] > 0]
        negative_graphs = [r for r in results if r['Difference'] < 0]
        zero_graphs = [r for r in results if r['Difference'] == 0]
        
        # Histogram
        differences = [r['Difference'] for r in results]
        plt.figure(figsize=(10, 6))
        plt.hist(differences, bins=20)
        plt.title('Histogram of Differences in Gate Count')
        plt.xlabel('Difference (Dynamic - Pauli)')
        plt.ylabel('Frequency')
        plt.savefig(f'../data/plots/differences_hist_{n_vertex}.png')
        plt.close()
        log_progress("Histogram plot saved")

        # Scatter plot
        plt.figure(figsize=(12, 6))
        colors = ['red' if d > 0 else 'green' if d < 0 else 'blue' 
                 for d in (r['Difference'] for r in results)]
        
        plt.scatter([r['Dynamic'] for r in results], 
                   [r['Pauli'] for r in results], 
                   c=colors, alpha=0.5)
        
        max_val = max(max(r['Dynamic'] for r in results),
                     max(r['Pauli'] for r in results))
        plt.plot([0, max_val], [0, max_val], 'k--', alpha=0.3)
        
        plt.xlabel('Dynamic Gate Count')
        plt.ylabel('Pauli Gate Count')
        plt.title('Dynamic vs Pauli Gate Counts')
        
        legend_elements = [
            plt.Rectangle((0,0),1,1, fc='green', alpha=0.5, label=f'Win ({len(negative_graphs)})'),
            plt.Rectangle((0,0),1,1, fc='red', alpha=0.5, label=f'Lose ({len(positive_graphs)})'),
            plt.Rectangle((0,0),1,1, fc='blue', alpha=0.5, label=f'Draw ({len(zero_graphs)})')
        ]
        plt.legend(handles=legend_elements)
        
        plt.savefig(f'../data/plots/gate_counts_scatter_{n_vertex}.png')
        plt.close()
        log_progress("Scatter plot saved")

        # Pie chart
        plt.figure(figsize=(12, 8))
        sizes = [len(negative_graphs), len(positive_graphs), 
                len(zero_graphs), len(error_graphs)]
        plt.pie(sizes,
               labels=['Win', 'Lose', 'Draw', 'Error'],
               colors=['green', 'red', 'blue', 'gray'],
               autopct=lambda pct: f'{pct:.1f}%\n({int(pct*total_graphs/100)})',
               startangle=90)
        
        plt.title(f'Distribution of Results\n(Total Graphs: {total_graphs})')
        
        # Add detailed legend
        plt.legend(
            [f'Win: Dynamic < Pauli ({len(negative_graphs)})',
             f'Lose: Dynamic > Pauli ({len(positive_graphs)})',
             f'Draw: Dynamic = Pauli ({len(zero_graphs)})',
             f'Error: Failed Processing ({len(error_graphs)})'],
            loc='center left',
            bbox_to_anchor=(1, 0.5)
        )
        
        plt.tight_layout()
        plt.savefig(f'../data/plots/results_pie_{n_vertex}.png', bbox_inches='tight')
        plt.close()
        log_progress("Pie chart saved")

        del positive_graphs, negative_graphs, zero_graphs

In [ ]:
# Parameters
n_qubits = 5
n_vertex = 2**n_qubits  
T = 200
delta_t = 0.1
BATCH_SIZE = 2

type = type = '50-50'

log_progress(f"Starting analysis: n_vertex={n_vertex}, T={T}, delta_t={delta_t}")

os.makedirs('../data/graphs', exist_ok=True)
os.makedirs('../data/plots', exist_ok=True)

try:
    with memory_cleanup():
        g6_file = f"./../data/graphs/{type}_graph{n_vertex}c.g6"
        log_progress(f"Reading input file: {g6_file}")
        
        results, error_graphs = process_graphs_in_batches(g6_file, n_vertex, BATCH_SIZE)
        log_progress("Graph processing complete")
        
        save_results(results, error_graphs, n_vertex)
        log_progress("Results saved")
        
        plot_results(results, error_graphs, n_vertex)
        log_progress("Plots generated")

    log_progress("Processing complete")
    
except Exception as e:
    log_progress(f"Fatal error: {str(e)}")
    raise